In [2]:
#Bibliotecas

import  pandas as pd

In [3]:
#1. Função de tratamento reutilizável para cada aba
def tratar_bases(df):
    # Copia o dataframe para evitar warnings
    df_limpa = df.copy()
    df_limpa.columns = df_limpa.columns.str.strip()
   # --- Limpeza de Colunas de Texto ---
    
    # Limpeza de NOME_PACIENTE: remove espaços extras e ajusta para Nome Próprio (Capitalized)
    if 'NOME_PACIENTE' in df_limpa.columns:
        df_limpa['NOME_PACIENTE'] = df_limpa['NOME_PACIENTE'].astype(str).str.strip().str.title()
        
    # Limpeza de PROCEDIMENTO: remove espaços extras e padroniza com Inicial Maiúscula
    if 'PROCEDIMENTO' in df_limpa.columns:
        df_limpa['PROCEDIMENTO'] = df_limpa['PROCEDIMENTO'].astype(str).str.strip().str.capitalize()
        
    # Limpeza e Padronização do CONVENIO
    if 'CONVENIO' in df_limpa.columns:
        # Padroniza texto removendo espaços nas pontas e convertendo para minúsculas para comparação
        df_limpa['CONVENIO'] = df_limpa['CONVENIO'].astype(str).str.strip()
        
        # Mapeamento para padronizar variações de grafia
        mapeamento_convenio = {
            'SulAmerica': 'Sul América',
            'Sul América': 'Sul América',
            'AMIL DENTAL': 'Amil Dental',
            'Amil Dental': 'Amil Dental',
            'amil dental': 'Amil Dental',
            'ODONTOPREV': 'OdontoPrev',
            'OdontoPrev': 'OdontoPrev',
            'odonto prev': 'OdontoPrev',
            'Bradesco Dental': 'Bradesco Dental',
            'Porto Seguro': 'Porto Seguro',
            'Particular': 'Particular'
        }
        df_limpa['CONVENIO'] = df_limpa['CONVENIO'].replace(mapeamento_convenio)
        
    # Limpeza de DENTISTA
    if 'DENTISTA' in df_limpa.columns:
        df_limpa['DENTISTA'] = df_limpa['DENTISTA'].astype(str).str.strip().str.title()
        
    # Limpeza do STATUS
    if 'STATUS' in df_limpa.columns:
        df_limpa['STATUS'] = df_limpa['STATUS'].astype(str).str.strip().str.capitalize()

    # --- Tipagem e Ajuste de Dados ---
    
    # Conversão da DATA_ATENDIMENTO para o formato datetime
    if 'DATA_ATENDIMENTO' in df_limpa.columns:
        df_limpa['DATA_ATENDIMENTO'] = pd.to_datetime(df_limpa['DATA_ATENDIMENTO'], errors='coerce')
        
    # Conversão de VALOR_SERVICO para numérico
    if 'VALOR_SERVICO' in df_limpa.columns:
        df_limpa['VALOR_SERVICO'] = pd.to_numeric(df_limpa['VALOR_SERVICO'], errors='coerce')
        
    return df_limpa 

In [4]:
# 2. Leitura do arquivo Excel contendo todas as abas
caminho_arquivo = "Base_Servicos_Odontologicos_2022_2025.xlsx"

# pd.read_excel com sheet_name=None lê TODAS as abas e retorna um dicionário {nome_da_aba: dataframe}
abas_dict = pd.read_excel(caminho_arquivo, sheet_name=None)

# Lista para armazenar cada aba tratada
lista_dfs_tratados = []

In [5]:
# 3. Processamento de cada aba (2022, 2023, 2024, 2025)
for nome_aba, df_aba in abas_dict.items():
    print(f"Tratando aba: {nome_aba}...")
    
    # Aplica a função de tratamento
    df_tratado = tratar_bases(df_aba)
    
    # Adiciona uma coluna identificando o ano/aba de origem (Boa prática de BI)
    df_tratado['ANO_ORIGEM'] = nome_aba
    
    # Adiciona à lista
    lista_dfs_tratados.append(df_tratado)

Tratando aba: 2022...
Tratando aba: 2023...
Tratando aba: 2024...
Tratando aba: 2025...


In [6]:
# 4. União (Concatenação) de todas as abas tratadas em uma única base
df_final = pd.concat(lista_dfs_tratados, ignore_index=True)

# Remove possíveis duplicatas gerais após a junção
df_final = df_final.drop_duplicates()

In [7]:
# 5. Exportação da base única tratada para Excel
nome_arquivo_saida = "Base_Servicos_Odontologicos_Consolidada.xlsx"
df_final.to_excel(nome_arquivo_saida, index=False)

print(f"\nProcesso concluído com sucesso! Base única salva em: {nome_arquivo_saida}")
print(f"Total de registros consolidados: {len(df_final)}")


Processo concluído com sucesso! Base única salva em: Base_Servicos_Odontologicos_Consolidada.xlsx
Total de registros consolidados: 20000
